In [ ]:
import json
import subprocess

import mlflow
import numpy as np
import requests

In [ ]:
# Start local MLflow server.
mlflow_server = subprocess.Popen(["mlflow", "server", "--host",  "127.0.0.1", "--port",  "8080"])

In [ ]:
# Set connection to the local MLflow server.
mlflow.set_tracking_uri("http://127.0.0.1:8080")
mlflow.set_experiment("/demo-experiment")

# Python Model

In [ ]:
class OutlierModel(mlflow.pyfunc.PythonModel):
    """Demo model that detects outliers."""

    def __init__(self, deviation=2):
        self._deviation = deviation

        # Learned parameters
        self.mean_ = None
        self.std_ = None

    def fit(self, X, y=None):
        """Fit the model to training dataset X."""
        self.mean_ = X.mean()
        self.std_ = X.std()
        return self

    def predict(self, context, X):
        """Predict whether X is an outlier."""
        lower = self.mean_ - self._deviation * self.std_
        upper = self.mean_ + self._deviation * self.std_

        return (X < lower) | (X > upper)


## Train

In [ ]:
# Create random training data.
train = np.random.randint(50, 100, 10)
train

In [ ]:
# Define a fixed test set.
test = np.array([40, 50, 60, 70, 80, 90, 100, 110])

In [ ]:
# Create and train a local model.
model = OutlierModel(deviation=1)
model = model.fit(train)

In [ ]:
# Metrics learned by the mode
print(f"""
Model learned metrics:
- mean:  {model.mean_:.2f}
- std:   {model.std_:.2f}
""")

In [ ]:
# Predict for the test set.
model.predict(None, test)

In [ ]:
# Training within an MLflow run.
with mlflow.start_run():

    # Set and log model parameters
    model_params = {"deviation": 1}
    mlflow.log_params(model_params)

    # Create a model.
    model = OutlierModel(**model_params)

    # Train the model.
    model.fit(train)

    # Log the learned model metrics.
    mlflow.log_metric("mean", model.mean_)
    mlflow.log_metric("std", model.std_)

    # Log the model itself.
    model_info = mlflow.pyfunc.log_model(
        python_model=model,
        artifact_path="demo-experiment",
    )

## Register

In [ ]:
# URI for the model trained in the last run
model_info.model_uri

In [ ]:
# Register the model from the last run.
registered = mlflow.register_model(
    model_uri=model_info.model_uri,
    name="OutlierModel",
    tags={"development-status": "alpha"}
)

## Load and Predict

In [ ]:
# Define which model and version to fetch.
model = "OutlierModel"
version = 1

# Load the MLflow model.
model_uri = f"models:/{model}/{version}"
loaded_model = mlflow.pyfunc.load_model(model_uri)

# Generate predictions.
loaded_model.predict(test)

## Building a Docker Image

In [ ]:
# Define which model and version to fetch.
model = "OutlierModel"
version = 1

In [ ]:
# Build the docker image.
# Note: Can take quite some time!
mlflow.models.build_docker(
    model_uri=f"models:/{model}/{version}",
    name='demo-experiment',
)

## Using the Docker container

In [ ]:
# Create the MFlow container.
mlflow_container = subprocess.Popen(["docker", "run", "-p", "8000:8080", "demo-experiment"])

In [ ]:
# Bring up the container with
requests.get("http://127.0.0.1:8000/ping")

In [ ]:
# Post some data to the Docker container.
response = requests.post(
    "http://127.0.0.1:8000/invocations",
    headers={"Content-Type": "application/json"},
    data=json.dumps({"inputs": [40, 50, 60, 70, 80, 90, 100, 110]})
)
response

In [ ]:
# Print the content of the response
response.content

In [ ]:
result = json.loads(response.content)
result["predictions"]

## Shut Down

In [ ]:
# Stop the local MLflow server.
mlflow_server.kill()

# Stop the local MLflow container
mlflow_container = subprocess.Popen(["docker", "stop", "demo-experiment"])
mlflow_container.kill()